# XAI Pipeline — EuroSAT multispectral CNN

13-band Sentinel-2 patches, 10 land-use classes (softmax).

All logic lives in `model/` and `xai/`. Train the model first with `python train_eurosat.py` from the repository root, then run this notebook top to bottom.

## 1 · Setup

In [ ]:
import os, sys, json
from collections import defaultdict
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

from model.runtime import setup
from xai.common import replace2linear
from xai.runner import ALL_METRICS, run_evaluation
from xai.visualization import plot_training_history, plot_cam_layers, plot_attribution_maps

from tensorflow.keras import mixed_precision
from tf_keras_vis.utils.scores import CategoricalScore
from model.cnn import Float32L2
from model.loaders.eurosat import EuroSATMultiTFLoader, preprocess_input
from xai import eurosat
from xai.common import get_background_samples
from xai.visualization import band_stretch_ranges, fixed_normalize

setup(42)
mixed_precision.set_global_policy('mixed_float16')

In [ ]:
DATA_DIR     = 'data/EuroSATallBands'
TFRECORD_DIR = 'data/tfrecords/eurosat'
MODEL_PATH   = 'models/eurosat.keras'
HISTORY_PATH = 'models/eurosat_history.json'
CSV_OUT      = 'results/eurosat_metrics.csv'

## 2 · Data

In [ ]:
with open(os.path.join(DATA_DIR, 'label_map.json')) as f:
    label_map = json.load(f)
class_names = sorted(label_map, key=label_map.get)

loaders = {split: EuroSATMultiTFLoader(csv_file=os.path.join(DATA_DIR, f'{split}.csv'), data_dir=DATA_DIR,
                                       augment=(split == 'train'), tfrecord_dir=TFRECORD_DIR)
           for split in ['train', 'validation', 'test']}
train_ds = loaders['train'].get_dataset(shuffle=True)
val_ds = loaders['validation'].get_dataset(shuffle=False)
test_ds = loaders['test'].get_dataset(shuffle=False)
loaders['train'].print_dataset_info(train_ds)

## 3 · Model

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH, custom_objects={'Float32L2': Float32L2})
model.summary()
with open(HISTORY_PATH) as f:
    plot_training_history(json.load(f))

## 4 · Classification performance

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
y_true, y_pred = [], []
for imgs, lbls in test_ds:
    y_true.extend(np.argmax(lbls.numpy(), axis=1))
    y_pred.extend(np.argmax(model.predict(imgs, verbose=0), axis=1))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion matrix')
plt.tight_layout(); plt.show()
print(classification_report(y_true, y_pred, target_names=class_names))

## 5 · XAI setup

In [ ]:
background = get_background_samples(train_ds, 100)
evaluator = eurosat.XAIEvaluator(model, fractions=10, robustness_n=10, sensitivity_iters=5, selectivity_patches=10)
eurosat.register_all(evaluator, background)
print('Explainers:', list(evaluator.explainers))

## 6 · Grad-CAM and Grad-CAM++ across layers

In [ ]:
img_array = tf.cast(preprocess_input(os.path.join(DATA_DIR, 'River/River_1.tif'),
                                     loaders['validation'].global_means, loaders['validation'].global_stds), tf.float32)
class_index = int(np.argmax(model.predict(img_array, verbose=0)))
print('Predicted:', class_names[class_index])
lo, hi = band_stretch_ranges(val_ds)
to_display = lambda img: fixed_normalize(np.asarray(img)[..., [3, 2, 1]], lo, hi)
plot_cam_layers(model, img_array, CategoricalScore(class_index), eurosat.LAYER_INDICES,
                to_display(img_array[0]), replace2linear)

## 7 · Attribution maps for confident predictions

In [ ]:
VIZ_CLASSES = ['Highway']

CONF_THRESH = 0.9
candidates = defaultdict(list)
for batch_imgs, _ in val_ds:
    batch_imgs = np.asarray(batch_imgs)
    preds = model.predict(batch_imgs, verbose=0)
    pred_idxs, confs = np.argmax(preds, axis=1), np.max(preds, axis=1)
    for img, pred_idx, conf in zip(batch_imgs, pred_idxs, confs):
        cls_name = class_names[pred_idx]
        if cls_name in VIZ_CLASSES and not candidates[cls_name] and CONF_THRESH <= conf <= CONF_THRESH + 0.11:
            candidates[cls_name].append((img, conf))
    if all(candidates[c] for c in VIZ_CLASSES):
        break
print({c: [round(float(conf), 3) for _, conf in v] for c, v in candidates.items()})

In [ ]:
for cls_name, items in candidates.items():
    for img, conf in items:
        plot_attribution_maps(evaluator, img, class_names.index(cls_name), to_display(img),
                              list(evaluator.explainers), f'{cls_name} (confidence {conf:.2f})')

## 8 · Quantitative evaluation

In [ ]:
images = eurosat.get_balanced_sample(test_ds, model, num_samples=2, threshold=0.7)
results = run_evaluation(evaluator, images, ALL_METRICS, None, CSV_OUT)
results